# Notebook 3: Monte Carlo 배치 실험 (v0.6b)
## 한국군 방공체계 M&S — 반복실험 및 통계 분석

이 노트북에서는:
- `modules/batch.py` BatchRunner를 사용한 병렬 배치 실행
- 시나리오(7) × 아키텍처(2) × 반복(300회) = 4,200회 Monte Carlo
- 12개 성능 메트릭 자동 수집 + 수렴성 검사
- 결과 CSV 저장 및 체크포인팅

In [ ]:
import sys, os, time, logging
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    !pip install simpy mesa networkx matplotlib pandas numpy scipy seaborn pillow tqdm --quiet
    from google.colab import drive
    drive.mount('/content/drive')
    sys.path.insert(0, '/content/drive/MyDrive/K-ADS_Simulation')
else:
    sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

from modules.config import EXPERIMENT_CONFIG, SCENARIO_PARAMS
from modules.batch import BatchRunner

# 로깅 설정
logging.basicConfig(level=logging.INFO, format='%(asctime)s %(message)s')

# 결과 저장 경로
RESULTS_DIR = 'results'
runner = BatchRunner(results_dir=RESULTS_DIR)

print('모듈 로드 완료!')
print(f'총 실행 수: {runner._total_runs()} '
      f'({EXPERIMENT_CONFIG["monte_carlo_runs"]} runs × '
      f'{len(EXPERIMENT_CONFIG["architectures"])} arch × '
      f'{len(EXPERIMENT_CONFIG["scenarios"])} scenarios)')

---
## 실험 설정

In [ ]:
# 실험 파라미터 확인
N_RUNS = EXPERIMENT_CONFIG['monte_carlo_runs']  # 300회
ARCHITECTURES = EXPERIMENT_CONFIG['architectures']  # ['linear', 'killweb']
SCENARIOS = EXPERIMENT_CONFIG['scenarios']  # 7개 시나리오
CHECKPOINT_INTERVAL = EXPERIMENT_CONFIG['checkpoint_interval']  # 50회

print(f'반복 횟수: {N_RUNS}')
print(f'아키텍처: {ARCHITECTURES}')
print(f'시나리오 ({len(SCENARIOS)}개):')
for s in SCENARIOS:
    print(f'  - {s}')
print(f'체크포인트 간격: {CHECKPOINT_INTERVAL}회')

---
## 소규모 파일럿 (10회)

In [ ]:
# 파일럿 실행 — 시간 추정 및 정상 동작 확인
df_pilot = runner.run_pilot(n=EXPERIMENT_CONFIG.get('pilot_runs', 10))

print(f'\n파일럿 완료: {len(df_pilot)}회')
print(f'\n--- 파일럿 결과 요약 ---')
pilot_metrics = ['leaker_rate', 'engagement_success_rate', 'sensor_to_shooter_time_mean']
available = [m for m in pilot_metrics if m in df_pilot.columns]
print(df_pilot.groupby(['scenario', 'architecture'])[available].describe().round(2))

---
## 본 실험 (300회 반복)

In [ ]:
# 본 실험 실행 — multiprocessing 병렬화
# n_workers를 조정하여 CPU 코어 수에 맞게 병렬화 (기본: min(cpu_count, 8))
df_results = runner.run_all()

print(f'\n실험 완료! {len(df_results)}개 결과')
print(f'저장 위치: {RESULTS_DIR}/monte_carlo_results.csv')

In [ ]:
# 결과 요약
print('=== 실험 결과 요약 ===')
summary_metrics = [
    'leaker_rate', 'engagement_success_rate',
    'sensor_to_shooter_time_mean', 'ammo_efficiency',
    'max_concurrent_engagements', 'c2_throughput',
    'multi_engagement_rate', 'avg_shooters_per_multi_engagement',
]
available = [m for m in summary_metrics if m in df_results.columns]
summary = df_results.groupby(['scenario', 'architecture'])[available].agg(
    ['mean', 'std']
).round(2)

print(summary)
summary.to_csv(os.path.join(RESULTS_DIR, 'experiment_summary.csv'))
print(f'\n요약 저장 → {RESULTS_DIR}/experiment_summary.csv')

In [ ]:
# 수렴성 검사 — BatchRunner.check_convergence() 활용
import matplotlib.pyplot as plt

metrics_to_check = ['leaker_rate', 'engagement_success_rate',
                     'sensor_to_shooter_time_mean', 'ammo_efficiency']
metric_labels = ['Leaker Rate (%)', 'Engagement Success Rate (%)',
                 'Sensor-to-Shooter Time (s)', 'Ammo Efficiency']

fig, axes = plt.subplots(2, 2, figsize=(16, 10))

for idx, (metric, label) in enumerate(zip(metrics_to_check, metric_labels)):
    ax = axes[idx // 2][idx % 2]
    for scenario in SCENARIOS:
        for arch in ARCHITECTURES:
            subset = df_results[
                (df_results['scenario'] == scenario) &
                (df_results['architecture'] == arch)
            ]
            conv = runner.check_convergence(subset, metric)
            cumulative_mean = conv['cumulative_means']
            if cumulative_mean:
                s_name = scenario.replace('scenario_', 'S').split('_')[0]
                style = '-' if arch == 'killweb' else '--'
                ax.plot(cumulative_mean, label=f'{s_name}-{arch}',
                       linewidth=1.5, linestyle=style)

    ax.set_xlabel('Runs')
    ax.set_ylabel(label)
    ax.set_title(f'Convergence: {label}')
    ax.legend(fontsize=7, ncol=2)
    ax.grid(True, alpha=0.3)

fig.suptitle('Monte Carlo Convergence Check (v0.6b)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'convergence_check.png'), dpi=150, bbox_inches='tight')
plt.show()

# 수렴 판정 결과 출력
print('\n=== Convergence Summary ===')
for metric in metrics_to_check:
    conv_all = runner.check_convergence(df_results, metric)
    status = 'CONVERGED' if conv_all['converged'] else 'NOT CONVERGED'
    print(f'{metric}: {status} (max_var={conv_all["max_variation"]:.6f})')